In [ ]:
# Imports and reproducibility
import numpy as np, pandas as pd
from pathlib import Path

rng = np.random.default_rng(42)
out = Path('.')

## Define our simple event schema
We keep this small and friendly: categorical fields (role, resource_type, etc.) and a few numeric fields we can plot and test for drift.

In [ ]:
users = [f'user_{i}' for i in range(530)]
roles = ['engineer','analyst','service','admin']
resource_types = ['code_repo','db_table','secrets_vault','build_system','analytics']
actions = ['read','write','delete','configure']
geo = ['NA','EU','APAC','LATAM']

def sample_base(n):
    return pd.DataFrame({
        'timestamp': rng.integers(1_700_000_000, 1_700_086_400, size=n),
        'user_id': rng.choice(users, size=n),
        'role': rng.choice(roles, size=n, p=[0.45,0.30,0.20,0.05]),
        'resource_type': rng.choice(resource_types, size=n),
        'action': rng.choice(actions, size=n, p=[0.55,0.25,0.05,0.15]),
        'bytes_transferred': np.round(rng.lognormal(mean=10, sigma=1, size=n)).astype(int),
        'hour_of_day': rng.integers(0,24,size=n),
        'geo_region': rng.choice(geo, size=n, p=[0.55,0.20,0.15,0.10]),
        'ip_reputation_score': rng.beta(8,2,size=n),
        'session_length_sec': np.round(rng.gamma(4, 120, size=n)).astype(int)
    })

## Generate baseline and inject a few anomaly patterns
We'll create:
- Burst outliers (huge bytes on secrets_vault writes)
- Backdoor trigger pattern (rare combo repeated)
- Slow drift (ip reputation distribution shifts for subset of users)
- Uniform noise block (simulated corrupted feed)

In [ ]:
N = 9000
base = sample_base(N)
base['anomaly_type'] = 'base'

# Burst outliers
burst = sample_base(120)
burst['bytes_transferred'] *= rng.integers(50,120,size=len(burst))
burst['resource_type'] = 'secrets_vault'
burst['action'] = 'write'
burst['anomaly_type'] = 'burst'

# Backdoor trigger
backdoor = sample_base(150)
backdoor['role'] = 'admin'
backdoor['resource_type'] = 'code_repo'
backdoor['action'] = 'configure'
backdoor['hour_of_day'] = 3
backdoor['anomaly_type'] = 'backdoor'

# Slow drift (ip_reputation_score becomes more uniform for subset)
drift = sample_base(180)
mask = drift['user_id'].isin(drift['user_id'].unique()[:40])
drift.loc[mask,'ip_reputation_score'] = rng.beta(2,2,size=mask.sum())
drift['anomaly_type'] = 'drift'

# Uniform noise block
noise = sample_base(100)
for col in ['bytes_transferred','ip_reputation_score','session_length_sec']:
    noise[col] = rng.uniform(low=0, high=max(1, noise[col].max())*1.2, size=len(noise))
noise['anomaly_type'] = 'noise'

dataset = pd.concat([base, burst, backdoor, drift, noise], ignore_index=True)
dataset = dataset.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle

## Save outputs
We save the full mixed dataset and also a baseline-only snapshot for training an unsupervised model later.

In [ ]:
dataset.to_csv(out / 'events_raw.csv', index=False)
base.to_csv(out / 'events_baseline.csv', index=False)
print('Saved:', (out/'events_raw.csv').resolve())
print('Saved:', (out/'events_baseline.csv').resolve())
dataset['anomaly_type'].value_counts()